In [6]:
import os
import re
import csv
from datetime import datetime

In [7]:
source = r"G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5"
destination = r"G:\c_out2"


In [8]:
def extract_quark_text(file_path):
    """Identifies QuarkXPress 3 files and extracts coherent text blocks."""
    try:
        with open(file_path, 'rb') as f:
            data = f.read()

        print (file_path)
        
        # Check for QuarkXPress 3 signature [cite: 1]
        if not data.startswith(b'MMXPR3'):
            return None

        # Extract printable sequences (MacRoman/ASCII) at least 15 chars long
        # This filters out small binary noise and formatting tags [cite: 10, 17]
        text_blobs = re.findall(b'[\x20-\x7E\xA0-\xFF]{15,}', data)
        
        clean_text = []
        # Keywords to skip (technical metadata in the file) [cite: 8, 10, 161]
        skip_keywords = ['LaserWriter', 'Century-Book', 'QRGB', 'QCMK', 'PnHx', 'TEXT']

        for blob in text_blobs:
            try:
                decoded = blob.decode('mac_roman').strip()
                if not any(k in decoded for k in skip_keywords):
                    # Clean up 'MenÕs' to 'Men's' etc.
                    clean_text.append(decoded)
            except UnicodeDecodeError:
                continue
                
        return "\n\n".join(clean_text)
    except Exception as e:
        return f"Error: {e}"

def run_inventory(source_dir, output_dir):
    """Processes files, creates individual TXTs, and one master CSV."""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    csv_path = os.path.join(output_dir, "inventory.csv")
    
    with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['file_name', 'folder', 'date_modified', 'size_kb', 'extracted_text']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for root, dirs, files in os.walk(source_dir):
            for filename in files:
                file_path = os.path.join(root, filename)
                
                # Get Metadata
                stats = os.stat(file_path)
                dt_mod = datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S')
                size_kb = round(stats.st_size / 1024, 2)
                
                # Extract Text
                content = extract_quark_text(file_path)
                
                if content:
                    # 1. Save individual text file
                    txt_name = f"{filename}.txt"
                    with open(os.path.join(output_dir, txt_name), 'w', encoding='utf-8') as tf:
                        tf.write(content)
                    
                    # 2. Add to CSV
                    writer.writerow({
                        'file_name': filename,
                        'folder': os.path.basename(root),
                        'date_modified': dt_mod,
                        'size_kb': size_kb,
                        'extracted_text': content[:500] + "..." # Truncate for CSV readability
                    })
                    print(f"Processed: {filename}")



In [9]:
#Set your paths here

run_inventory(source, destination)

G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\gide-- Let’s Consider
G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\Spt - Three-peat (Westport )
G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\Spt - Summer (NFL Coaches)
G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\Spt - Seniors (Southeast)
G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\Spt - Offensive Line (Chiefs)
G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\Spt - New Coach (Central)
G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\Spt - NHL Report (Diversity)
G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\Spt - NHL Report
G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\Spt - NCAA Raises
G:\c_out\Call 09-05\Pages and Pics\Call Editorial-Pics\CALL COPY SEPT. 5\Spt - Monday Niters 9_5
G:\c_out\Call 09-05\Pa